# 03 — A/B Testing
Treat **cellular** as the treatment group and **telephone** as the control group.
Run hypothesis tests, calculate lift, and report statistical significance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, norm
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/processed/cleaned_data.csv')
if 'contact' not in df.columns:
    df['contact'] = np.where(df.get('contact_cellular', pd.Series(0)) == 1, 'cellular', 'telephone')

treatment = df[df['contact'] == 'cellular'].copy()
control   = df[df['contact'] == 'telephone'].copy()
print(f'Treatment (cellular): {len(treatment):,}')
print(f'Control (telephone):  {len(control):,}')

## 1. Conversion Rate Summary

In [ ]:
t_conv = treatment['y'].mean()
c_conv = control['y'].mean()
lift = (t_conv - c_conv) / c_conv * 100

summary = pd.DataFrame({
    'Group': ['Cellular (Treatment)', 'Telephone (Control)'],
    'N': [len(treatment), len(control)],
    'Conversions': [treatment['y'].sum(), control['y'].sum()],
    'Conv Rate (%)': [round(t_conv*100, 2), round(c_conv*100, 2)]
})
print(summary.to_string(index=False))
print(f'\nLift = {lift:.1f}%  (cellular over telephone)')

## 2. Chi-Square Test — Is the Difference Significant?

In [ ]:
contingency = pd.crosstab(df['contact'], df['y'])
print('Contingency Table:')
print(contingency)

chi2, p_chi2, dof, expected = chi2_contingency(contingency)
print(f'\nChi-square statistic : {chi2:.4f}')
print(f'p-value              : {p_chi2:.6f}')
print(f'Degrees of freedom   : {dof}')
sig = 'SIGNIFICANT' if p_chi2 < 0.05 else 'NOT significant'
print(f'\nResult: {sig} at 95% confidence (α = 0.05)')

## 3. Confidence Interval for Lift

In [ ]:
def proportion_ci(p, n, z=1.96):
    se = np.sqrt(p * (1 - p) / n)
    return p - z * se, p + z * se

t_lo, t_hi = proportion_ci(t_conv, len(treatment))
c_lo, c_hi = proportion_ci(c_conv, len(control))

print(f'Cellular  95% CI: [{t_lo*100:.2f}%, {t_hi*100:.2f}%]')
print(f'Telephone 95% CI: [{c_lo*100:.2f}%, {c_hi*100:.2f}%]')

# Visualise CIs
fig, ax = plt.subplots(figsize=(7, 4))
groups = ['Cellular\n(Treatment)', 'Telephone\n(Control)']
means  = [t_conv*100, c_conv*100]
errs   = [(t_conv - t_lo)*100, (c_conv - c_lo)*100]
ax.bar(groups, means, yerr=errs, capsize=8, color=['#4C72B0','#DD8452'], edgecolor='white', alpha=0.85)
ax.set_title('Conversion Rate with 95% Confidence Intervals', fontsize=13)
ax.set_ylabel('Conversion Rate (%)')
plt.tight_layout()
plt.savefig('../reports/figures/ab_ci.png', dpi=150)
plt.show()

## 4. t-Test — Balance: Converters vs Non-Converters

In [ ]:
conv_bal    = df[df['y']==1]['age']      # 'age' as proxy; swap for balance if available
nonconv_bal = df[df['y']==0]['age']

t_stat, p_ttest = ttest_ind(conv_bal, nonconv_bal, equal_var=False)
print(f'Converters   mean age: {conv_bal.mean():.1f}')
print(f'Non-converters mean age: {nonconv_bal.mean():.1f}')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value    : {p_ttest:.6f}')
print('Result:', 'SIGNIFICANT difference' if p_ttest < 0.05 else 'No significant difference')

## 5. Power Analysis — Was Our Sample Large Enough?

In [ ]:
from math import ceil

alpha = 0.05
power = 0.80
p1 = t_conv  # treatment rate
p2 = c_conv  # control rate
p_avg = (p1 + p2) / 2

z_alpha = norm.ppf(1 - alpha / 2)
z_beta  = norm.ppf(power)
n_required = ceil(
    (z_alpha * np.sqrt(2 * p_avg * (1 - p_avg)) + z_beta * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2
    / (p1 - p2)**2
)

print(f'Minimum sample size per group for 80% power: {n_required:,}')
print(f'Actual treatment group size: {len(treatment):,}')
print(f'Actual control group size  : {len(control):,}')
print('\n✓ Sample is adequately powered.' if min(len(treatment), len(control)) >= n_required else '⚠ Under-powered — interpret with caution.')

## 6. Summary Statement

In [ ]:
print('=' * 60)
print('A/B TEST SUMMARY')
print('=' * 60)
print(f'  Treatment (Cellular)  : {t_conv*100:.2f}% conversion rate')
print(f'  Control   (Telephone) : {c_conv*100:.2f}% conversion rate')
print(f'  Lift                  : {lift:.1f}%')
print(f'  Chi-square p-value    : {p_chi2:.6f}')
print(f'  Statistically sig?    : {"YES" if p_chi2 < 0.05 else "NO"} (α=0.05)')
print('=' * 60)
print(f'\n→ Cellular channel delivered {lift:.1f}% higher conversion rate')
print(f'  than telephone at 95% confidence (p = {p_chi2:.4f}).')